# Pair classifier analysis

Loads saved outputs from `src/task1/xgboost_pair_classifier.py` and regenerates the analysis plots.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

CLASS_NAMES = ["LOS+NLOS", "NLOS+NLOS"]

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    if (PROJECT_ROOT.parent / "src").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
    else:
        raise FileNotFoundError("Could not locate project root containing src/")

PAIR_DIR = PROJECT_ROOT / "outputs" / "pair_classifier"
ARTIFACT_DIR = PAIR_DIR / "artifacts"
DATA_DIR = PROJECT_ROOT / "data" / "preprocessed"

plt.style.use("seaborn-v0_8-darkgrid")
PAIR_DIR

In [ ]:
y_test = np.load(ARTIFACT_DIR / "y_test_pair.npy")
y_pred = np.load(ARTIFACT_DIR / "y_pred_xgb.npy")
y_proba = np.load(ARTIFACT_DIR / "y_proba_xgb.npy")
cm = np.load(ARTIFACT_DIR / "pair_confusion_matrix.npy")
roc_data = np.load(ARTIFACT_DIR / "pair_roc_curve_data.npz")
feature_importance = pd.read_csv(PAIR_DIR / "pair_feature_importance_xgb.csv")
category_importance = pd.read_csv(PAIR_DIR / "pair_feature_importance_by_category_xgb.csv")
error_summary = pd.read_csv(ARTIFACT_DIR / "pair_error_summary.csv")
top_error_features = pd.read_csv(ARTIFACT_DIR / "pair_top_error_features.csv")

display(feature_importance.head())
display(category_importance)
display(error_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=ax,
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    linewidths=0.5,
)
ax.set_title("Pair-Level Confusion Matrix - XGBoost", fontsize=13, fontweight="bold")
ax.set_xlabel("Predicted Pair", fontsize=11)
ax.set_ylabel("Actual Pair", fontsize=11)

total = cm.sum()
for i in range(2):
    for j in range(2):
        ax.text(
            j + 0.5,
            i + 0.75,
            f"({cm[i, j] / max(total, 1) * 100:.1f}%)",
            ha="center",
            va="center",
            fontsize=9,
            color="red",
        )

plt.tight_layout()
plt.savefig(PAIR_DIR / "pair_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

fpr = roc_data["fpr"]
tpr = roc_data["tpr"]
auc = float(np.trapezoid(tpr, fpr))

fig, ax = plt.subplots(figsize=(9, 7))
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", linewidth=1.5, label="Random Classifier (AUC = 0.500)")
ax.plot(fpr, tpr, color="#e67e22", linewidth=2.5, label=f"XGBoost (AUC = {auc:.4f})")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("Pair-Level ROC Curve - XGBoost", fontsize=13, fontweight="bold")
ax.legend(fontsize=10, loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PAIR_DIR / "pair_roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
top20 = feature_importance.head(20).copy()

def feature_color(name: str) -> str:
    if name.startswith("PEAK2"):
        return "#e74c3c"
    if name.startswith("CIR"):
        return "#3498db"
    return "#95a5a6"

colors = [feature_color(name) for name in top20["Feature"]]
fig, ax = plt.subplots(figsize=(12, 8))
y_pos = np.arange(len(top20))
ax.barh(y_pos, top20["Importance"], color=colors, alpha=0.85)
ax.set_yticks(y_pos)
ax.set_yticklabels(top20["Feature"], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Feature Importance (Gain)", fontsize=12)
ax.set_title("Pair Classifier - Top 20 Feature Importance (XGBoost)", fontsize=12, fontweight="bold")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(PAIR_DIR / "pair_feature_importance_xgb.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(category_importance["Category"], category_importance["Importance"], color=["#e74c3c", "#3498db", "#95a5a6"], alpha=0.85)
for bar, val in zip(bars, category_importance["Importance"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003, f"{val * 100:.1f}%", ha="center", fontsize=11, fontweight="bold")
ax.set_ylabel("Total Importance", fontsize=12)
ax.set_ylim(0, max(category_importance["Importance"].max() * 1.2, 1.0))
ax.set_title("Pair Classifier - Feature Importance by Category (XGBoost)", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PAIR_DIR / "pair_importance_by_category.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

In [ ]:
boundary_low = float(error_summary.loc[0, "boundary_low"])
boundary_high = float(error_summary.loc[0, "boundary_high"])
error_mask = y_pred != y_test
error_probs = y_proba[error_mask]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), squeeze=False)
ax = axes[0, 0]
if error_probs.size:
    ax.hist(error_probs, bins=20, color="#e67e22", alpha=0.85)
    ax.axvspan(boundary_low, boundary_high, color="#3498db", alpha=0.15, label="Boundary region")
    ax.legend()
ax.set_title("Misclassified sample probabilities", fontweight="bold")
ax.set_xlabel("Predicted probability for class 1")
ax.set_ylabel("Count")

ax = axes[0, 1]
top_error = top_error_features.head(10).iloc[::-1]
if not top_error.empty:
    ax.barh(top_error["feature"], top_error["effect"], color="#8e44ad", alpha=0.85)
ax.set_title("Top features distinguishing errors", fontweight="bold")
ax.set_xlabel("Absolute mean difference")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

thresholds = np.linspace(0.05, 0.95, 200)
f1_scores = []
precision_scores = []
recall_scores = []
accuracy_scores = []

for t in thresholds:
    preds = (y_proba >= t).astype(int)
    f1_scores.append(f1_score(y_test, preds))
    precision_scores.append(precision_score(y_test, preds, zero_division=0))
    recall_scores.append(recall_score(y_test, preds, zero_division=0))
    accuracy_scores.append(accuracy_score(y_test, preds))

best_f1_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_f1_idx]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds, f1_scores, color="#e67e22", linewidth=2.5, label="F1-Score")
ax.plot(thresholds, precision_scores, color="#2ecc71", linewidth=1.8, linestyle="--", label="Precision")
ax.plot(thresholds, recall_scores, color="#3498db", linewidth=1.8, linestyle="--", label="Recall")
ax.plot(thresholds, accuracy_scores, color="#9b59b6", linewidth=1.8, linestyle=":", label="Accuracy")
ax.axvline(x=0.5, color="gray", linestyle=":", alpha=0.6, label="Default threshold (0.5)")
ax.axvline(x=best_threshold, color="#e74c3c", linewidth=2, linestyle="-", label=f"Best F1 threshold ({best_threshold:.3f})")
ax.scatter([best_threshold], [f1_scores[best_f1_idx]], color="#e74c3c", s=100, zorder=5)
ax.set_xlabel("Decision Threshold", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Decision Threshold Optimization Curve", fontsize=13, fontweight="bold")
ax.legend(fontsize=10, loc="lower left")
ax.set_xlim(0.05, 0.95)
ax.set_ylim(0.6, 1.05)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PAIR_DIR / "pair_threshold_optimization.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print(f"Best F1 threshold: {best_threshold:.3f}")
print(f"  F1       : {f1_scores[best_f1_idx]:.4f}")
print(f"  Precision: {precision_scores[best_f1_idx]:.4f}")
print(f"  Recall   : {recall_scores[best_f1_idx]:.4f}")
print(f"  Accuracy : {accuracy_scores[best_f1_idx]:.4f}")
print(f"\nDefault (0.5) comparison:")
default_idx = np.argmin(np.abs(thresholds - 0.5))
print(f"  F1       : {f1_scores[default_idx]:.4f}")
print(f"  Precision: {precision_scores[default_idx]:.4f}")
print(f"  Recall   : {recall_scores[default_idx]:.4f}")
print(f"  Accuracy : {accuracy_scores[default_idx]:.4f}")